In [ ]:
%load_ext autoreload 
#hopefully this will reload the modules when they are changed specifically when i change the plotting modules
%autoreload 2

In [ ]:
from myproject import GalaxyGroup, Subhalo, ListGalaxyGroup, AstroPlotter
import h5py as h5
import numpy as np

In [ ]:
sim = 'TNG300-1'
scratchDataDirc = f'/scratch/poulin.al/lopsided/{sim}/data'
scratchPlotDirc = f'/scratch/poulin.al/lopsided/{sim}/plots'
localDataDirc = f'/Users/alexpoulin/Library/CloudStorage/OneDrive-NortheasternUniversity/TGB–Data'


In [ ]:
generalRewrite = False

## load data from hdf5 file


In [ ]:
data_file = scratchDataDirc + f'/galaxy_data_{sim}.hdf5'
# data_file = localDataDirc + f'/galaxy_data_{sim}.hdf5'
with h5.File(data_file, 'r') as f:
    list_of_galaxy_groups = ListGalaxyGroup.from_hdf5(f)
print(f'Loaded galaxy data from {data_file}')

In [ ]:
# print(f"len filtered: {len(filtered_galaxy_groups)}")
filtered_list_of_galaxy_groups = list_of_galaxy_groups.getFilterSubhalos(minGGMass=1e14)

In [ ]:
#check the number of galaxy groups loaded
print(f'Number of galaxy groups loaded: {filtered_list_of_galaxy_groups.getNumGalaxyGroups()}')
print(f"Number of subhalos in GG 0: {filtered_list_of_galaxy_groups.getGalaxyGroupI(0).getNumSubhalos()}")
print(f"Number of subhalos in GG 10: {filtered_list_of_galaxy_groups.getGalaxyGroupI(10).getNumSubhalos()}")
print(f' Range of satellites: {filtered_list_of_galaxy_groups.getRangeOfNumSubhalos()}')

## Make plots for Pairwise Polar Differences

In [ ]:
list_pairwise_polar_differences = filtered_list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar', rewrite=generalRewrite)

In [ ]:
# polar_bin_centers, pairwise_polar_differences = list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(parallelize=False, tempSaveDir=localDataDirc)
polar_bin_centers, pairwise_polar_differences_binned, pairwise_polar_errorbars = ListGalaxyGroup.get_histogram_bins(list_pairwise_polar_differences, bins = np.arange(0, 185, 10), parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar', errorbarType='bootstrap')
print('Computed pairwise polar differences between satellite galaxies.')
# print(polar_bin_centers)
#may need:
# #check and flatten any tuples
        # for galaxyPairwiseGroup in self.list_pairwise_differences:
        #     for pair in galaxyPairwiseGroup:
        #         # print(f"pair: {pair, type(pair)}")
        #         if isinstance(pair, tuple):
        #             print(f"Warning.... flattening tuple pair: {pair, type(pair)}")
        #             return
        #             pairwise_differences_flatten.extend(pair)
        #         else:
        #             pairwise_differences_flatten.append(pair)

In [ ]:
prob_polar_plotter = AstroPlotter()
prob_polar_plotter.scatter_plot(
    polar_bin_centers, 
    list_pairwise_polar_differences,
    errorBars = pairwise_polar_errorbars,
    xlabel='Pairwise Polar Difference (degrees)',
    ylabel='Probability Density',
    title=f'Probability Pairwise Polar Difference Distribution for {sim}',
    # ylim = (0, 0.01),
    output_filename=scratchPlotDirc + f'/pairwise_polar/pairwise_polar_difference_{sim}.png',
    grid=True,
)

In [ ]:
# save bin centers and probabilities to text file
output_data_file = scratchDataDirc + f'/pairwise_polar/pairwise_polar_difference_{sim}.txt'
# print(f"data to be saved: {np.column_stack((polar_bin_centers, pairwise_polar_differences))}")
np.savetxt(output_data_file, np.column_stack((polar_bin_centers, list_pairwise_polar_differences)), header='Pairwise Polar Difference (degrees)    Probability Density')

## Make plots for Mean Resultant Length (MRL) directionality

In [ ]:
MRL_values = filtered_list_of_galaxy_groups.compute_probablity_distribution_of_MRL_directionality(parallelize=False, tempSaveDir=f'{scratchDataDirc}/MRL_directionality', rewrite=generalRewrite)

In [ ]:
print(len(filtered_list_of_galaxy_groups.MRL_values))

In [ ]:
MRL_bin_centers, MRL_directionality, MRL_errorbars = filtered_list_of_galaxy_groups.get_histogram_bins(MRL_values, bin_size=0.005, binLow=0, binHigh=1, errorbarType='bootstrap')

In [ ]:
prob_MRL_plotter = AstroPlotter()
prob_MRL_plotter.scatter_plot(
    MRL_bin_centers, 
    MRL_directionality,
    errorBars = MRL_errorbars,
    xlabel='MRL Directionality',
    ylabel='Probability Density',
    title=f'Probability MRL Directionality Distribution for {sim}',
    # ylim=(0,4)
    output_filename=scratchPlotDirc + f'/MRL/MRL_directionality_{sim}.png',
    grid=True
)

In [ ]:
#save bin centers and probabilities to text file
output_data_file_MRL = scratchDataDirc + f'/MRL_directionality/MRL_directionality_{sim}.txt'
# print(f"data to be saved: {np.column_stack((MRL_bin_centers, MRL_directionality))}")
np.savetxt(output_data_file_MRL, np.column_stack((MRL_bin_centers, MRL_directionality)), header='MRL Directionality    Probability Density')

In [ ]:
#save bin MRL values to text file
output_data_file_MRL = scratchDataDirc + f'/MRL_directionality/MRL_directionality_RAW_{sim}.txt'
# print(f"data to be saved: {np.vstack(filtered_list_of_galaxy_groups.MRL_values)}")
np.savetxt(output_data_file_MRL, np.vstack(filtered_list_of_galaxy_groups.MRL_values), header='MRL Directionality')

## make plots for red vs blue galaxies

In [ ]:
filtered_red_list_of_galaxy_groups = filtered_list_of_galaxy_groups.getFilterSubhalos(redGalaxies=True)
print(f'Number of galaxy groups with only red satellites: {filtered_red_list_of_galaxy_groups.getRangeOfNumSubhalos()}')

filtered_blue_list_of_galaxy_groups = filtered_list_of_galaxy_groups.getFilterSubhalos(blueGalaxies=True)
print(f'Number of galaxy groups with only blue satellites: {filtered_blue_list_of_galaxy_groups.getRangeOfNumSubhalos()}')


list_pairwise_polar_differences_red = filtered_red_list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_color/pairwise_polar_red', rewrite=generalRewrite)
list_pairwise_polar_differences_blue = filtered_blue_list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_color/pairwise_polar_blue', rewrite=generalRewrite)


In [ ]:
polar_bin_centers_red, pairwise_polar_differences_red, pairwise_polar_red_errorbars = ListGalaxyGroup.get_histogram_bins(list_pairwise_polar_differences_red, bins = np.arange(0, 185, 10), parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_red', errorbarType='bootstrap')
polar_bin_centers_blue, pairwise_polar_differences_blue, pairwise_polar_blue_errorbars = ListGalaxyGroup.get_histogram_bins(list_pairwise_polar_differences_blue, bins = np.arange(0, 185, 10), parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_blue', errorbarType='bootstrap')

In [ ]:
print('Computed pairwise polar differences between red and blue satellite galaxies.')
prob_polar_red_blue_plotter = AstroPlotter()
prob_polar_red_blue_fig, prob_polar_red_blue_ax = prob_polar_red_blue_plotter.create_figure()
prob_polar_red_blue_plotter.scatter_plot(
    polar_bin_centers_red, 
    pairwise_polar_differences_red,
    ax=prob_polar_red_blue_ax, # Use the same axis for overlay
    xlabel='Pairwise Polar Difference (degrees)',
    ylabel='Probability Density',
    title=f'Probability Pairwise Polar Difference Distribution for Red and Blue Satellites in {sim}',
    # ylim = (0, 0.01),
    c=['red'],
    errorBars = pairwise_polar_red_errorbars,
    label = "Red Galaxies (g-r) ≥ 0.65",
    output_filename=None,  # Disable saving for combined plot
    grid=True,
)
prob_polar_red_blue_plotter.scatter_plot(
    polar_bin_centers_blue, 
    pairwise_polar_differences_blue,
    ax=prob_polar_red_blue_ax,  # Use the same axis for overlay
    xlabel='Pairwise Polar Difference (degrees)',
    ylabel='Probability Density',
    title=f'Probability Pairwise Polar Difference Distribution for Red and Blue Satellites in {sim}',
    # ylim = (0, 0.01),
    c=['blue'],
    errorBars = pairwise_polar_blue_errorbars,
    label = "Blue Galaxies (g-r) < 0.65",
    include_legend=True,
    output_filename=scratchPlotDirc + f'/pairwise_polar/pairwise_polar_difference_red_blue_{sim}.png',
    grid=True,
)

In [ ]:
# save as txt file
output_data_file = scratchDataDirc + f'/pairwise_polar_color/pairwise_polar_color_difference_{sim}.txt'
print(f"data to be saved: {np.column_stack((polar_bin_centers_red, pairwise_polar_differences_red, pairwise_polar_differences_blue))}")
np.savetxt(output_data_file, np.column_stack((polar_bin_centers_red, pairwise_polar_differences_red, pairwise_polar_differences_blue)), header='Pairwise Polar Difference (degrees)    Probability Density (Red)    Probability Density (Blue)')

## make plots for 150 vs 50 member galaxies

In [ ]:
filtered_GT150_list_of_galaxy_groups = filtered_list_of_galaxy_groups.getFilterSubhalos(minNumGalaxies=150)
print(f'Number of galaxy groups with more than 150 satellites: {filtered_GT150_list_of_galaxy_groups.getNumGalaxyGroups()}')
filtered_LT50_list_of_galaxy_groups = filtered_list_of_galaxy_groups.getFilterSubhalos(maxNumGalaxies=50)
print(f'Number of galaxy groups with less than 50 satellites: {filtered_LT50_list_of_galaxy_groups.getNumGalaxyGroups()}')

list_pairwise_polar_differences_GT150 = filtered_GT150_list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_memberNum/pairwise_polar_GT150', rewrite=generalRewrite)
list_pairwise_polar_differences_LT50 = filtered_LT50_list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_memberNum/pairwise_polar_LT50', rewrite=generalRewrite)

In [ ]:
print('Computed pairwise polar differences for galaxy groups with >150 and <50 satellites.')
polar_bin_centers_GT150, pairwise_polar_differences_GT150, pairwise_polar_GT150_errorbars = ListGalaxyGroup.get_histogram_bins(list_pairwise_polar_differences_GT150, bins = np.arange(0, 185, 10), parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_GT150', errorbarType='bootstrap')
polar_bin_centers_LT50, pairwise_polar_differences_LT50, pairwise_polar_LT50_errorbars = ListGalaxyGroup.get_histogram_bins(list_pairwise_polar_differences_LT50, bins = np.arange(0, 185, 10), parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_LT50', errorbarType='bootstrap')

In [ ]:
prob_polar_GT150_LT50_plotter = AstroPlotter()
prob_polar_GT150_LT50_fig, prob_polar_GT150_LT50_ax = prob_polar_GT150_LT50_plotter.create_figure()
prob_polar_GT150_LT50_plotter.scatter_plot(
    polar_bin_centers_GT150, 
    pairwise_polar_differences_GT150,
    errorBars = pairwise_polar_GT150_errorbars,
    ax=prob_polar_GT150_LT50_ax, # Use the same axis for overlay
    xlabel='Pairwise Polar Difference (degrees)',
    ylabel='Probability Density',
    title=f'Probability Pairwise Polar Difference Distribution for >150 and <50 Satellites in {sim}',
    # ylim = (0, 0.01),
    label="More than 150 members",
    output_filename=None,  # Disable saving for combined plot
    grid=True,
)
prob_polar_GT150_LT50_plotter.scatter_plot(
    polar_bin_centers_LT50, 
    pairwise_polar_differences_LT50,
    errorBars = pairwise_polar_LT50_errorbars,
    ax=prob_polar_GT150_LT50_ax,  # Use the same axis for overlay
    xlabel='Pairwise Polar Difference (degrees)',
    ylabel='Probability Density',
    title=f'Probability Pairwise Polar Difference Distribution for >150 and <50 Satellites in {sim}',
    # ylim = (0, 0.01),
    label="Less than 50 members",
    include_legend=True,
    output_filename=scratchPlotDirc + f'/pairwise_polar/pairwise_polar_difference_GT150_LT50_{sim}.png',
    grid=True,
)

In [ ]:
# save as text file
output_data_file = scratchDataDirc + f'/pairwise_polar_memberNum/pairwise_polar_memberNum_difference_{sim}.txt'
# print(f"data to be saved: {np.column_stack((polar_bin_centers_LT50, pairwise_polar_differences_LT50, pairwise_polar_differences_GT150))}")
np.savetxt(output_data_file, np.column_stack((polar_bin_centers_LT50, pairwise_polar_differences_LT50, pairwise_polar_differences_GT150)), header='Pairwise Polar Difference (degrees)    Probability Density (<50 Satellites)    Probability Density (>150 Satellites)')

## make plots for <35% vs >65% R200 plots

In [ ]:
filtered_LT35R200_list_of_galaxy_groups = filtered_list_of_galaxy_groups.getFilterSubhalos(withinXPercentR200=[0,0.35])
print(f'Number of galaxy groups with satellites within 35% R200: {filtered_LT35R200_list_of_galaxy_groups.getNumGalaxyGroups()}')
filtered_GT65R200_list_of_galaxy_groups = filtered_list_of_galaxy_groups.getFilterSubhalos(withinXPercentR200=[0.65,1.00])
print(f'Number of galaxy groups with satellites within 65-100% R200: {filtered_GT65R200_list_of_galaxy_groups.getNumGalaxyGroups()}')

list_pairwise_polar_differences_LT35R200 = filtered_LT35R200_list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_radius/pairwise_polar_LT35R200', rewrite=generalRewrite)
list_pairwise_polar_differences_GT65R200 = filtered_GT65R200_list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_radius/pairwise_polar_GT65R200', rewrite=generalRewrite)

In [ ]:
polar_bin_centers_LT35R200, pairwise_polar_differences_LT35R200, pairwise_polar__LT35R400_errorbars = filtered_LT35R200_list_of_galaxy_groups.get_histogram_bins(list_pairwise_polar_differences_LT35R200, bins = np.arange(0, 185, 10), parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_LT35R200', errorbarType='bootstrap')
polar_bin_centers_GT65R200, pairwise_polar_differences_GT65R200, pairwise_polar_GT65R200_errorbars = filtered_GT65R200_list_of_galaxy_groups.get_histogram_bins(list_pairwise_polar_differences_GT65R200, bins = np.arange(0, 185, 10), parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_GT65R200', errorbarType='bootstrap')

In [ ]:
prob_polar_LT35R200_GT65R200_plotter = AstroPlotter()
prob_polar_LT35R200_GT65R200_fig, prob_polar_LT35R200_GT65R200_ax = prob_polar_LT35R200_GT65R200_plotter.create_figure()
prob_polar_LT35R200_GT65R200_plotter.scatter_plot(
    polar_bin_centers_LT35R200, 
    pairwise_polar_differences_LT35R200,
    errorBars = pairwise_polar__LT35R400_errorbars,
    ax=prob_polar_LT35R200_GT65R200_ax, # Use the same axis for overlay
    xlabel='Pairwise Polar Difference (degrees)',
    ylabel='Probability Density',
    title=f'Probability Pairwise Polar Difference Distribution for 65%<r<100% R200 and 0%<r<35% R200 in {sim}',
    # ylim = (0, 0.01),
    label="0<r<0.35 R200",
    output_filename=None,  # Disable saving for combined plot
    grid=True,
)
prob_polar_LT35R200_GT65R200_plotter.scatter_plot(
    polar_bin_centers_GT65R200, 
    pairwise_polar_differences_GT65R200,
    errorBars = pairwise_polar_GT65R200_errorbars,
    ax=prob_polar_LT35R200_GT65R200_ax,  # Use the same axis for overlay
    xlabel='Pairwise Polar Difference (degrees)',
    ylabel='Probability Density',
    title=f'Probability Pairwise Polar Difference Distribution for \n 65%<r<100% R200 and 0%<r<35% R200 in {sim}',
    # ylim = (0, 0.01),
    label="0.65<r<1.00 R200",
    include_legend=True,
    output_filename=scratchPlotDirc + f'/pairwise_polar/pairwise_polar_difference_LT35R200_GT65R200_{sim}.png',
    grid=True,
)

In [ ]:
# save as text file
output_data_file = scratchDataDirc + f'/pairwise_polar_radius/pairwise_polar_radius_difference_{sim}.txt'
# print(f"data to be saved: {np.column_stack((polar_bin_centers_LT35R200, pairwise_polar_differences_LT35R200, pairwise_polar_differences_GT65R200))}")
np.savetxt(output_data_file, np.column_stack((polar_bin_centers_LT35R200, pairwise_polar_differences_LT35R200, pairwise_polar_differences_GT65R200)), header='Pairwise Polar Difference (degrees)    Probability Density (<35% R200)    Probability Density (65-100% R200)')

## check central-FoF distance offests

In [ ]:
#save txt file if most massive != most central
#with lines:
# 1: x_cm of the halo
# 2: y_cm of the halo
# 3: z_cm of the halo
# 4: offset of the "most central galaxy" from the halo center of mass
# 5: ratio of the total masses of the most central and most massive galaxoes
# 6: offset of the most massive galaxy from the halo center of mass

with open(scratchDataDirc + f'/most_massive_vs_most_central/most_massive_vs_most_central_{sim}.txt', 'w') as f:
    f.write('x_cm    y_cm    z_cm    offset_most_central    mass_ratio    offset_most_massive most_massive_r_ratio_r200\n')
    
for galaxy_group in list_of_galaxy_groups.getAllGalaxyGroups():
    if galaxy_group.getMostMassiveSubhalo().getIdx() != galaxy_group.getMostCentralSubhalo().getIdx():
        output_data_file = scratchDataDirc + f'/most_massive_vs_most_central/most_massive_vs_most_central_{sim}.txt'
        with open(output_data_file, 'a') as f:
            x_cm, y_cm, z_cm = galaxy_group.getPos()
            most_central_pos = galaxy_group.getMostCentralSubhalo().getPosition()
            most_massive_pos = galaxy_group.getMostMassiveSubhalo().getPosition()
            offset_most_central = np.linalg.norm(most_central_pos - galaxy_group.getPos())
            offset_most_massive = np.linalg.norm(most_massive_pos - galaxy_group.getPos())
            mass_ratio = galaxy_group.getMostCentralSubhalo().getStellarMass() / galaxy_group.getMostMassiveSubhalo().getStellarMass()
            most_massive_r_ratio_r200 = np.linalg.norm(galaxy_group.getMostMassiveSubhalo().getPosition()) / galaxy_group.getRCrit200()
            f.write(f"{x_cm} {y_cm} {z_cm} {offset_most_central} {mass_ratio} {offset_most_massive} {most_massive_r_ratio_r200} \n")

In [ ]:
#filter to groups with stellar mass < 10^{14} Msun
filtered_gt13_ls14_list_of_galaxy_groups = list_of_galaxy_groups.getFilterSubhalos(maxGGMass=1e14)

In [ ]:
#check the number of galaxy groups loaded
print(f'Number of galaxy groups loaded: {filtered_gt13_ls14_list_of_galaxy_groups.getNumGalaxyGroups()}')
print(f"Number of subhalos in GG 0: {filtered_gt13_ls14_list_of_galaxy_groups.getGalaxyGroupI(0).getNumSubhalos()}")
print(f"Number of subhalos in GG 10: {filtered_gt13_ls14_list_of_galaxy_groups.getGalaxyGroupI(10).getNumSubhalos()}")
print(f' Range of satellites: {filtered_gt13_ls14_list_of_galaxy_groups.getRangeOfNumSubhalos()}')

In [ ]:
def getCentral_FoF_distanceOffset(list_of_galaxy_groups : ListGalaxyGroup):
    distanceOffsets = []
    central_positions = []
    FoF_positions = []
    for gg in list_of_galaxy_groups.listGalaxyGroups:
        central_subhalo = gg.getCentralSubhalo()
        fof_center = gg.getPos()
        central_positions.append(central_subhalo.getPosition())
        FoF_positions.append(fof_center)
        
        distanceOffsets.append(np.linalg.norm(central_subhalo.getPosition() - fof_center))
    return np.array(distanceOffsets), np.array(central_positions), np.array(FoF_positions)

In [ ]:
distances_gt13_ls14 = getCentral_FoF_distanceOffset(filtered_gt13_ls14_list_of_galaxy_groups)[0]
print(distances_gt13_ls14)
distances_gt14 = getCentral_FoF_distanceOffset(filtered_list_of_galaxy_groups)[0]


In [ ]:
central_FoF_Plotter = AstroPlotter()
central_FoF_fig, central_FoF_ax = central_FoF_Plotter.create_figure()
central_FoF_Plotter.histogram(
    distances_gt13_ls14,
    bins=60, #np.logspace(np.log10(0.1), np.log10(5000), 60),
    ax = central_FoF_ax,  # Use the same axis for overlay
    xlabel='Distance between Central Subhalo and Galaxy Group Position (kpc)',
    ylabel='Density of Galaxy Groups',
    title=f'Distance between Central Subhalo and Galaxy Group Position for {sim} ($M_{{200}}$<1e14 Msun)',
    label='1e13<$M_{{200}}$<1e14 Msun',
    ylog=True,
    legend=True,
    linealpha=0.5,
    output_filename=scratchPlotDirc + f'/central_disparities/central_FoF_distance_{sim}_lt14.png',
    percentage=True,
    grid=True,
)
central_FoF_Plotter.histogram(
    distances_gt14,
    bins=60, #np.logspace(np.log10(0.1), np.log10(5000), 60),
    ax = central_FoF_ax,  # Use the same axis for overlay
    xlabel='Distance between Central Subhalo and Galaxy Group Position (kpc)',
    ylabel='Density of Galaxy Groups',
    title=f'Distance between Central Subhalo and Galaxy Group Position for {sim}',
    label='$M_{{200}}$>1e14 Msun',
    ylog=True,
    legend=True,
    linealpha=0.5,
    output_filename=scratchPlotDirc + f'/central_disparities/central_FoF_distance_{sim}_gt14.png',
    percentage=True,
    grid=True,
)

In [ ]:
#make pairwise polar difference plots for these two mass bins as well
list_pairwise_polar_differences_13_14 = filtered_gt13_ls14_list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(parallelize=False)
list_pairwise_polar_differences_gt14 = filtered_list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(parallelize=False)

In [ ]:
polar_bin_centers_13_14, pairwise_polar_differences_13_14, pairwise_polar_13_14_errorbars = ListGalaxyGroup.get_histogram_bins(list_pairwise_polar_differences_13_14, bins = np.arange(0, 185, 10), parallelize=False, errorbarType='bootstrap')

In [ ]:
polar_bin_centers_gt14, pairwise_polar_differences_gt14, pairwise_polar_gt14_errorbars = ListGalaxyGroup.get_histogram_bins(list_pairwise_polar_differences_gt14, bins = np.arange(0, 185, 10), parallelize=False, errorbarType='bootstrap')

In [ ]:
prob_polar_gt14_lt14_plotter = AstroPlotter()
prob_polar_gt14_lt14_fig, prob_polar_gt14_lt14_ax = prob_polar_gt14_lt14_plotter.create_figure()
prob_polar_gt14_lt14_plotter.scatter_plot(
    polar_bin_centers_13_14, 
    pairwise_polar_differences_13_14,
    errorBars = pairwise_polar_13_14_errorbars,
    ax=prob_polar_gt14_lt14_ax, # Use the same axis for overlay
    xlabel='Pairwise Polar Difference (degrees)',
    ylabel='Probability Density',
    title=f'Probability Pairwise Polar Difference Distribution for 1e13<$M_{{200}}$<1e14 and $M_{{200}}$>1e14 Msun in {sim}',
    # ylim = (0, 0.01),
    label="1e13<$M_{{200}}$<1e14 Msun",
    output_filename=None,  # Disable saving for combined plot
    grid=True,
)
prob_polar_gt14_lt14_plotter.scatter_plot(
    polar_bin_centers_gt14, 
    pairwise_polar_differences_gt14,
    errorBars = pairwise_polar_gt14_errorbars,
    ax=prob_polar_gt14_lt14_ax,  # Use the same axis for        overlay
    xlabel='Pairwise Polar Difference (degrees)',
    ylabel='Probability Density',
    title=f'Probability Pairwise Polar Difference Distribution for 1e13<$M_{{200}}$<1e14 and $M_{{200}}$>1e14 Msun in {sim}',
    # ylim = (0, 0.01),
    label="$M_{{200}}$>1e14 Msun",
    include_legend=True,
    output_filename=scratchPlotDirc + f'/pairwise_polar/pairwise_polar_difference_gt14_lt14_{sim}.png',
    grid=True,
)

## prob distribution of mass of five subgroups
13-13.5, 13.5-14, 14-14.5, 14.5-15, >15

In [ ]:
#filter to groups with stellar mass 10^{13} < $M_{{200}}$ < 10^{13.5} Msun
filtered_13_13p5_list_of_galaxy_groups = list_of_galaxy_groups.getFilterSubhalos(maxGGMass=5e13, minGGMass=1e13)

#filter to groups with stellar mass 10^{13.5} < $M_{{200}}$ < 10^{14} Msun
filtered_13p5_14_list_of_galaxy_groups = list_of_galaxy_groups.getFilterSubhalos(maxGGMass=1e14, minGGMass=5e13)

#filter to groups with stellar mass 10^{14} < $M_{{200}}$ < 10^{14.5} Msun
filtered_14_14p5_list_of_galaxy_groups = list_of_galaxy_groups.getFilterSubhalos(maxGGMass=5e14, minGGMass=1e14)

#filter to groups with stellar mass 10^{14.5} < $M_{{200}}$ < 10^{15} Msun
filtered_14p5_15_list_of_galaxy_groups = list_of_galaxy_groups.getFilterSubhalos(maxGGMass=1e15, minGGMass=5e14)

#filter to groups with stellar mass $M_{{200}}$ > 10^{15} Msun
filtered_gt15_list_of_galaxy_groups = list_of_galaxy_groups.getFilterSubhalos(minGGMass=1e15)


In [ ]:
print(f'Number of galaxy groups 13-13.5 loaded: {filtered_13_13p5_list_of_galaxy_groups.getNumGalaxyGroups()}')
print(f'Number of galaxy groups 13.5-14 loaded: {filtered_13p5_14_list_of_galaxy_groups.getNumGalaxyGroups()}')
print(f'Number of galaxy groups 14-14.5 loaded: {filtered_14_14p5_list_of_galaxy_groups.getNumGalaxyGroups()}')
print(f'Number of galaxy groups 14.5-15 loaded: {filtered_14p5_15_list_of_galaxy_groups.getNumGalaxyGroups()}')
print(f'Number of galaxy groups >15 loaded: {filtered_gt15_list_of_galaxy_groups.getNumGalaxyGroups()}')

print(f"total number of galaxy groups: {filtered_13_13p5_list_of_galaxy_groups.getNumGalaxyGroups() + filtered_13p5_14_list_of_galaxy_groups.getNumGalaxyGroups() + filtered_14_14p5_list_of_galaxy_groups.getNumGalaxyGroups() + filtered_14p5_15_list_of_galaxy_groups.getNumGalaxyGroups() + filtered_gt15_list_of_galaxy_groups.getNumGalaxyGroups()}")

#print the num of galaxies in each group where the central is not the most massive
num_mostMassiveNotCentral = 0
for gg in filtered_13_13p5_list_of_galaxy_groups.listGalaxyGroups:
    if gg.getMostMassiveSubhalo().getIdx() != gg.getMostCentralSubhalo().getIdx():
        # print(f"GG ID: {gg.getGroupID()}, Num of galaxies: {gg.getNumSubhalos()}")
        num_mostMassiveNotCentral +=1
    else:
        print(gg.getMostMassiveSubhalo().getPosition(), gg.getMostCentralSubhalo().getPosition())
print(f"Number of groups with most massive not central in 13-13.5 bin: {num_mostMassiveNotCentral}")
num_mostMassiveNotCentral = 0
for gg in filtered_13p5_14_list_of_galaxy_groups.listGalaxyGroups:
    if gg.getMostMassiveSubhalo().getIdx() != gg.getMostCentralSubhalo().getIdx():
        # print(f"GG ID: {gg.getGroupID()}, Num of galaxies: {gg.getNumSubhalos()}")
        num_mostMassiveNotCentral +=1
print(f"Number of groups with most massive not central in 13.5-14 bin: {num_mostMassiveNotCentral}")
num_mostMassiveNotCentral = 0
for gg in filtered_14_14p5_list_of_galaxy_groups.listGalaxyGroups:
    if gg.getMostMassiveSubhalo().getIdx() != gg.getMostCentralSubhalo().getIdx():
        # print(f"GG ID: {gg.getGroupID()}, Num of galaxies: {gg.getNumSubhalos()}")
        num_mostMassiveNotCentral +=1
print(f"Number of groups with most massive not central in 14-14.5 bin: {num_mostMassiveNotCentral}")
num_mostMassiveNotCentral = 0
for gg in filtered_14p5_15_list_of_galaxy_groups.listGalaxyGroups:
    if gg.getMostMassiveSubhalo().getIdx() != gg.getMostCentralSubhalo().getIdx():
        # print(f"GG ID: {gg.getGroupID()}, Num of galaxies: {gg.getNumSubhalos()}")
        num_mostMassiveNotCentral +=1
print(f"Number of groups with most massive not central in 14.5-15 bin: {num_mostMassiveNotCentral}")
num_mostMassiveNotCentral = 0
for gg in filtered_gt15_list_of_galaxy_groups.listGalaxyGroups:
    if gg.getMostMassiveSubhalo().getIdx() != gg.getMostCentralSubhalo().getIdx():
        # print(f"GG ID: {gg.getGroupID()}, Num of galaxies: {gg.getNumSubhalos()}")
        num_mostMassiveNotCentral +=1
print(f"Number of groups with most massive not central in >15 bin: {num_mostMassiveNotCentral}")
        

In [ ]:
#plot the mass ratio distribution (most central / most massive) for these mass bins
prob_mass_ratio_plotter = AstroPlotter()
prob_mass_ratio_fig, prob_mass_ratio_ax = prob_mass_ratio_plotter.create_figure()
bins = np.linspace(0.4, 1, 30)
prob_mass_ratio_plotter.histogram(
    [gg.getMostCentralSubhalo().getStellarMass() / gg.getMostMassiveSubhalo().getStellarMass() for gg in filtered_13_13p5_list_of_galaxy_groups.getAllGalaxyGroups()],
    bins=bins,
    ax = prob_mass_ratio_ax,
    xlabel='Mass Ratio between Most Central and Most Massive Subhalo',
    ylabel='Density of Galaxy Groups',
    title=f'Mass Ratio between Most Central and Most Massive Subhalo for {sim}',
    label='1e13<$M_{{200}}$<1e13.5 Msun',
    ylog=True,
    legend=True,
    linealpha=0.5,
    output_filename=None,
    percentage=True,
    grid=True,
)
prob_mass_ratio_plotter.histogram(
    [gg.getMostCentralSubhalo().getStellarMass() / gg.getMostMassiveSubhalo().getStellarMass() for gg in filtered_13p5_14_list_of_galaxy_groups.getAllGalaxyGroups()],
    bins=bins,
    ax = prob_mass_ratio_ax,
    xlabel='Mass Ratio between Most Central and Most Massive Subhalo',
    ylabel='Density of Galaxy Groups',
    title=f'Mass Ratio between Most Central and Most Massive Subhalo for {sim}',
    label='1e13.5<$M_{{200}}$<1e14 Msun',
    ylog=True,
    legend=True,
    linealpha=0.5,
    output_filename=None,
    percentage=True,
    grid=True,
)
prob_mass_ratio_plotter.histogram(
    [gg.getMostCentralSubhalo().getStellarMass() / gg.getMostMassiveSubhalo().getStellarMass() for gg in filtered_14_14p5_list_of_galaxy_groups.getAllGalaxyGroups()],
    bins=bins,
    ax = prob_mass_ratio_ax,
    xlabel='Mass Ratio between Most Central and Most Massive Subhalo',
    ylabel='Density of Galaxy Groups',
    title=f'Mass Ratio between Most Central and Most Massive Subhalo for {sim}',
    label='1e14<$M_{{200}}$<1e14.5 Msun',
    ylog=True,
    legend=True,
    linealpha=0.5,
    output_filename=None,
    percentage=True,
    grid=True,
)
prob_mass_ratio_plotter.histogram(
    [gg.getMostCentralSubhalo().getStellarMass() / gg.getMostMassiveSubhalo().getStellarMass() for gg in filtered_14p5_15_list_of_galaxy_groups.getAllGalaxyGroups()],
    bins=bins,
    ax = prob_mass_ratio_ax,
    xlabel='Mass Ratio between Most Central and Most Massive Subhalo',
    ylabel='Density of Galaxy Groups',
    title=f'Mass Ratio between Most Central and Most Massive Subhalo for {sim}',
    label='1e14.5<$M_{{200}}$<1e15 Msun',
    ylog=True,
    legend=True,
    linealpha=0.5,
    output_filename=None,
    percentage=True,
    grid=True,
)
prob_mass_ratio_plotter.histogram(
    [gg.getMostCentralSubhalo().getStellarMass() / gg.getMostMassiveSubhalo().getStellarMass() for gg in filtered_gt15_list_of_galaxy_groups.getAllGalaxyGroups()],
    bins=bins,
    ax = prob_mass_ratio_ax,
    xlabel='Mass Ratio ($M_{{central}} / M_{{most massive}}$)',
    ylabel='Density of Galaxy Groups',
    title=f'Mass Ratio between Most Central and Most Massive Subhalo for {sim}',
    label='$M_{{200}}$>1e15 Msun',
    ylog=True,
    legend=True,
    linealpha=0.5,
    output_filename=scratchPlotDirc + f'/central_disparities/most_massive_vs_most_central_massRatio_{sim}_massBins.png',
    percentage=True,
    grid=True,
)

In [ ]:
# plot the r/r200 of the most massive subhalo for these mass bins
prob_massive_r_ratio_plotter = AstroPlotter()
prob_massive_r_ratio_fig, prob_massive_r_ratio_ax = prob_massive_r_ratio_plotter.create_figure()
bins = np.linspace(0, 1, 30)
prob_massive_r_ratio_plotter.histogram(
    [np.linalg.norm(gg.getMostMassiveSubhalo().getPosition()) / gg.getRCrit200() for gg in filtered_13_13p5_list_of_galaxy_groups.getAllGalaxyGroups()],
    bins=bins,
    ax = prob_massive_r_ratio_ax,
    xlabel='r/r200 of Most Massive Subhalo',
    ylabel='Density of Galaxy Groups',
    title=f'r/r200 of Most Massive Subhalo for {sim}',
    label='1e13<$M_{{200}}$<1e13.5 Msun',
    ylog=True,
    legend=True,
    linealpha=0.5,
    output_filename=None,
    percentage=True,
    grid=True,
)
prob_massive_r_ratio_plotter.histogram(
    [np.linalg.norm(gg.getMostMassiveSubhalo().getPosition()) / gg.getRCrit200() for gg in filtered_13p5_14_list_of_galaxy_groups.getAllGalaxyGroups()],
    bins=bins,
    ax = prob_massive_r_ratio_ax,
    xlabel='r/r200 of Most Massive Subhalo',
    ylabel='Density of Galaxy Groups',
    title=f'r/r200 of Most Massive Subhalo for {sim}',
    label='1e13.5<$M_{{200}}$<1e14 Msun',
    ylog=True,
    legend=True,
    linealpha=0.5,
    output_filename=None,
    percentage=True,
    grid=True,
)
prob_massive_r_ratio_plotter.histogram(
    [np.linalg.norm(gg.getMostMassiveSubhalo().getPosition()) / gg.getRCrit200() for gg in filtered_14_14p5_list_of_galaxy_groups.getAllGalaxyGroups()],
    bins=bins,
    ax = prob_massive_r_ratio_ax,
    xlabel='r/r200 of Most Massive Subhalo',
    ylabel='Density of Galaxy Groups',
    title=f'r/r200 of Most Massive Subhalo for {sim}',
    label='1e14<$M_{{200}}$<1e14.5 Msun',
    ylog=True,
    legend=True,
    linealpha=0.5,
    output_filename=None,
    percentage=True,
    grid=True,
)
prob_massive_r_ratio_plotter.histogram(
    [np.linalg.norm(gg.getMostMassiveSubhalo().getPosition()) / gg.getRCrit200() for gg in filtered_14p5_15_list_of_galaxy_groups.getAllGalaxyGroups()],
    bins=bins,
    ax = prob_massive_r_ratio_ax,
    xlabel='r/r200 of Most Massive Subhalo',
    ylabel='Density of Galaxy Groups',
    title=f'r/r200 of Most Massive Subhalo for {sim}',
    label='1e14.5<$M_{{200}}$<1e15 Msun',
    ylog=True,
    legend=True,
    linealpha=0.5,
    output_filename=None,
    percentage=True,
    grid=True,
)
prob_massive_r_ratio_plotter.histogram(
    [np.linalg.norm(gg.getMostMassiveSubhalo().getPosition()) / gg.getRCrit200() for gg in filtered_gt15_list_of_galaxy_groups.getAllGalaxyGroups()],
    bins=bins,
    ax = prob_massive_r_ratio_ax,
    xlabel='r/r200 of Most Massive Subhalo',
    ylabel='Density of Galaxy Groups',
    title=f'r/r200 of Most Massive Subhalo for {sim}',
    label='$M_{{200}}$>1e15 Msun',
    ylog=True,
    legend=True,
    linealpha=0.5,
    output_filename=scratchPlotDirc + f'/central_disparities/most_massive_r_ratio_r200_{sim}_massBins.png',
    percentage=True,
    grid=True,
)

In [ ]:
# plot the median number of subhalos within r200 for these mass bins
def plot_median_num_subhalos_by_mass_bins(listGG : list[tuple[ListGalaxyGroup, str]]):
    median_subhalo_plotter = AstroPlotter()
    median_subhalo_fig, median_subhalo_ax = median_subhalo_plotter.create_figure()
    bins = np.linspace(0, 500, 100)
    mass_bin_labels = []
    for i, (mass_filtered_list, mass_bin_label) in enumerate(listGG):
        median_num_subhalos = []
        for galaxy_group in mass_filtered_list.getAllGalaxyGroups():
            median_num = len(galaxy_group.getSatelliteSubhalos())
            median_num_subhalos.append(median_num)
        mass_bin_labels.append(mass_bin_label)
        print(f"Mass bin: {mass_bin_label}, median number of subhalos: {median_num}")
    
        median_subhalo_plotter.histogram(
            median_num_subhalos,
            bins=bins,
            ax= median_subhalo_ax,
            xlabel='Number of Satellite Subhalos within r200',
            ylabel='Density of Galaxy Groups',
            title=f'Number of Satellite Subhalos within r200 for {sim}',
            label=mass_bin_label,
            ylog=True,
            legend=True,
            linealpha=0.5,
            output_filename=None,
            percentage=True,
            grid=True,
        )
    median_subhalo_plotter.save_figure(
        fig=median_subhalo_fig,
        filename=scratchPlotDirc + f'/median_num_subhalos_by_mass_bins_{sim}.png'
    )

In [ ]:
plot_median_num_subhalos_by_mass_bins([(filtered_13_13p5_list_of_galaxy_groups, '1e13<$M_{{200}}$<1e13.5 Msun'),
                                       (filtered_13p5_14_list_of_galaxy_groups, '1e13.5<$M_{{200}}$<1e14 Msun'),
                                       (filtered_14_14p5_list_of_galaxy_groups, '1e14<$M_{{200}}$<1e14.5 Msun'),
                                       (filtered_14p5_15_list_of_galaxy_groups, '1e14.5<$M_{{200}}$<1e15 Msun'),
                                       (filtered_gt15_list_of_galaxy_groups, '$M_{{200}}$>1e15 Msun')]) 

In [ ]:
#plot the pairwise polar differences for these mass bins and split between central is most massive or not
def plot_pairwise_polar_by_mass_bins_and_centralMassive_status(listGG : list[tuple[ListGalaxyGroup, str]]): #mass_filtered_list : ListGalaxyGroup, mass_bin_label):
    prob_polar_mass_plotter = AstroPlotter()
    prob_polar_mass_fig, prob_polar_mass_ax = prob_polar_mass_plotter.create_figure(ncols=1, nrows=len(listGG), figsize=(8, 6*len(listGG)))

    colors = ['blue', 'orange', 'red', 'green', 'purple']                                                          
    for i, (mass_filtered_list, mass_bin_label) in enumerate(listGG):
        central_most_massive_list_of_galaxy_groups = mass_filtered_list.getFilterSubhalos(centralIsMostMassive=True)
        not_central_most_massive_list_of_galaxy_groups = mass_filtered_list.getFilterSubhalos(centralIsMostMassive=False)
        print(f"len GG: {len(central_most_massive_list_of_galaxy_groups.getAllGalaxyGroups()), len(not_central_most_massive_list_of_galaxy_groups.getAllGalaxyGroups())}")

        list_pairwise_polar_differences_centralMassive = central_most_massive_list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(parallelize=False)
        list_pairwise_polar_differences_notCentralMassive = not_central_most_massive_list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(parallelize=False)
        list_pairwise_polar_differences_total = mass_filtered_list.compute_probablity_distribution_of_polar_differences(parallelize=False)
        
        polar_bin_centers_centralMassive, pairwise_polar_differences_centralMassive, pairwise_polar_centralMassive_errorbars = ListGalaxyGroup.get_histogram_bins(list_pairwise_polar_differences_centralMassive, bins = np.arange(0, 185, 10), parallelize=False, errorbarType='bootstrap')
        polar_bin_centers_notCentralMassive, pairwise_polar_differences_notCentralMassive, pairwise_polar_notCentralMassive_errorbars = ListGalaxyGroup.get_histogram_bins(list_pairwise_polar_differences_notCentralMassive, bins = np.arange(0, 185, 10), parallelize=False, errorbarType='bootstrap')
        polar_bin_centers_total, pairwise_polar_differences_total, pairwise_polar_total_errorbars = ListGalaxyGroup.get_histogram_bins(list_pairwise_polar_differences_total, bins = np.arange(0, 185, 10), parallelize=False, errorbarType='bootstrap')
    
        if len(listGG) > 1:
            print(f"choosing ax, {i}")
            axToPlot = prob_polar_mass_ax[i]
        else:
            axToPlot = prob_polar_mass_ax
        print(f"len: {len(polar_bin_centers_centralMassive), len(polar_bin_centers_notCentralMassive), len(polar_bin_centers_total)}")
        print(f"first 5 values centralMassive: {pairwise_polar_differences_centralMassive[:5]}, notCentralMassive: {pairwise_polar_differences_notCentralMassive[:5]}, total: {pairwise_polar_differences_total[:5]}")

        prob_polar_mass_plotter.scatter_plot(
            polar_bin_centers_centralMassive, 
            pairwise_polar_differences_centralMassive,
            # errorBars = pairwise_polar_centralMassive_errorbars,
            ax=axToPlot, # Use the same axis for overlay
            overlay_color=colors[0],
            # xlabel='Pairwise Polar Difference (degrees)',
            # ylabel='Probability Density',
            # title=f'Probability Pairwise Polar Difference Distribution for {sim} ({mass_bin_label})',
            # ylim = (0, 0.01),
            label="Central is Most Massive",
            # output_filename=None,  # Disable saving for combined plot
            # grid=True,
        )
        prob_polar_mass_plotter.scatter_plot(
            polar_bin_centers_notCentralMassive, 
            pairwise_polar_differences_notCentralMassive,
            # errorBars = pairwise_polar_notCentralMassive_errorbars,
            ax=axToPlot,  # Use the same axis for overlay
            overlay_color=colors[1], 
            # xlabel='Pairwise Polar Difference (degrees)',
            # ylabel='Probability Density',
            # title=f'Probability Pairwise Polar Difference Distribution for {sim} ({mass_bin_label})',
            # ylim = (0, 0.01),
            label="Central is NOT Most Massive",
            # include_legend=True,
            # output_filename=None,
            # grid=True,
        )
        prob_polar_mass_plotter.scatter_plot(
            polar_bin_centers_total, 
            pairwise_polar_differences_total,
            # errorBars = pairwise_polar_total_errorbars,
            ax=axToPlot,  # Use the same axis for overlay
            overlay_color=colors[2],
            xlabel='Pairwise Polar Difference (degrees)',
            ylabel='Probability Density',
            title=f'Probability Pairwise Polar Difference Distribution for {sim} ({mass_bin_label})',
            # ylim = (0, 0.01),
            s=10,
            
            label="All Galaxy Groups",
            include_legend=True,
            # output_filename=scratchPlotDirc + f'/pairwise_polar_difference_total_{mass_bin_label.replace("<","lt").replace(">","gt")}_{sim}.png',
            grid=True,
        )

        
    return prob_polar_mass_plotter, prob_polar_mass_fig, prob_polar_mass_ax

In [ ]:
# run for each mass bin
plot_pairwise_polar_by_mass_bins_and_centralMassive_status([(filtered_13_13p5_list_of_galaxy_groups, '1e13<$M_{{200}}$<1e13.5 Msun'),
                                                            (filtered_13p5_14_list_of_galaxy_groups, '1e13.5<$M_{{200}}$<1e14 Msun'),
                                                            (filtered_14_14p5_list_of_galaxy_groups, '1e14<$M_{{200}}$<1e14.5 Msun'),
                                                            (filtered_14p5_15_list_of_galaxy_groups, '1e14.5<$M_{{200}}$<1e15 Msun'),
                                                            (filtered_gt15_list_of_galaxy_groups, '$M_{{200}}$>1e15 Msun')])
# plot_pairwise_polar_by_mass_bins_and_centralMassive_status([
#                                                             (filtered_gt15_list_of_galaxy_groups, '$M_{{200}}$>1e15 Msun')])

In [ ]:
#plot for all galaxy groups with central most massive vs not central most massive without splitting into mass bins
prob_polar_mass_plotter, prob_polar_mass_fig, prob_polar_mass_ax = plot_pairwise_polar_by_mass_bins_and_centralMassive_status([(list_of_galaxy_groups, '$M_{{200}}$>1e13 Msun')])

In [ ]:
#replot this with different ylims
prob_polar_mass_ax.set_ylim(0.0050, 0.0065)
prob_polar_mass_plotter.save_figure(prob_polar_mass_fig, scratchPlotDirc + f'/pairwise_polar_difference_allMasses_{sim}_differentYlim.png')

#overlay
# polar_bin_centers, pairwise_polar_differences, pairwise_polar_errorbars
prob_polar_mass_ax.errorbar(polar_bin_centers, list_pairwise_polar_differences, yerr=pairwise_polar_errorbars, fmt='o', label='All Galaxy Groups', color='black', markersize=3, alpha=0.5)
prob_polar_mass_ax.legend()
prob_polar_mass_plotter.save_figure(prob_polar_mass_fig, scratchPlotDirc + f'/pairwise_polar_difference_allMasses_{sim}_overlay.png')

## create MRL random distribution curves

In [ ]:
MRL_20_values = ListGalaxyGroup.compute_an_MRL_distribution_curves(parallelize=False, num_samples=10000, num_non_centrals=20)
MRL_20, MRL_20_bin_edges, MRL_20_errorbars = ListGalaxyGroup.get_histogram_bins(MRL_20_values, bin_size=0.005, binLow=0, binHigh=1, errorbarType='poisson')

MRL_50_values = ListGalaxyGroup.compute_an_MRL_distribution_curves(parallelize=False, num_samples=10000, num_non_centrals=50)
MRL_50, MRL_50_bin_edges, MRL_50_errorbars = ListGalaxyGroup.get_histogram_bins(MRL_50_values, bin_size=0.005, binLow=0, binHigh=1, errorbarType='poisson')

MRL_100_values = ListGalaxyGroup.compute_an_MRL_distribution_curves(parallelize=False, num_samples=10000, num_non_centrals=100)
MRL_100, MRL_100_bin_edges, MRL_100_errorbars = ListGalaxyGroup.get_histogram_bins(MRL_100_values, bin_size=0.005, binLow=0, binHigh=1, errorbarType='poisson')

MRL_200_values = ListGalaxyGroup.compute_an_MRL_distribution_curves(parallelize=False, num_samples=10000, num_non_centrals=200)
MRL_200, MRL_200_bin_edges, MRL_200_errorbars = ListGalaxyGroup.get_histogram_bins(MRL_200_values, bin_size=0.005, binLow=0, binHigh=1, errorbarType='poisson')

print(len(MRL_20), len(MRL_20_bin_edges))
print(MRL_20[MRL_20 > 0])
#plot with spline curve
MRL_plotter = AstroPlotter()
MRL_fig, MRL_ax = MRL_plotter.create_figure()
MRL_plotter.scatter_plot(
    MRL_20_bin_edges, 
    MRL_20,
    # errorBars = MRL_20_errorbars,
    ax=MRL_ax, # Use the same axis for overlay
    # ylim = (0, 0.01),
    label="20 Non-Centrals",
    alpha=0,
    s=50,
    output_filename=None,  # Disable saving for combined plot
    grid=True,
    spline_curvature=True,
)
#plot a small verticle line at 99th percentile of MRL_20_values
percentile_99_MRL_20 = np.percentile(MRL_20_values, 99)
MRL_ax.axvline(percentile_99_MRL_20, color='blue', linestyle='--', label='99th Percentile (20 Non-Centrals)')

MRL_plotter.scatter_plot(
    MRL_50_bin_edges, 
    MRL_50,
    # errorBars = MRL_50_errorbars,
    ax=MRL_ax,  # Use the same axis for overlay
    # ylim = (0, 0.01),
    label="50 Non-Centrals",
    alpha=0,
    output_filename=None,
    grid=True,
    spline_curvature=True,
)
#plot a small verticle line at 99th percentile of MRL_50_values
percentile_99_MRL_50 = np.percentile(MRL_50_values, 99)
MRL_ax.axvline(percentile_99_MRL_50, color='orange', linestyle='--', label='99th Percentile (50 Non-Centrals)')

MRL_plotter.scatter_plot(
    MRL_100_bin_edges, 
    MRL_100,
    # errorBars = MRL_100_errorbars,
    ax=MRL_ax,  # Use the same axis for overlay
    # ylim = (0, 0.01),
    label="100 Non-Centrals",
    alpha=0,
    output_filename=None,
    grid=True,
    spline_curvature=True,
)
#plot a small verticle line at 99th percentile of MRL_100_values
percentile_99_MRL_100 = np.percentile(MRL_100_values, 99)
MRL_ax.axvline(percentile_99_MRL_100, color='red', linestyle='--', label='99th Percentile (100 Non-Centrals)')

MRL_plotter.scatter_plot(
    MRL_200_bin_edges, 
    MRL_200,
    # errorBars = MRL_500_errorbars,
    ax=MRL_ax,  # Use the same axis for overlay
    xlabel='MRL Directionality',
    ylabel='Probability Density',
    title=f'MRL Distribution Curves',
    # ylim = (0, 0.01),
    label="200 Non-Centrals",
    alpha=0,
    output_filename=scratchPlotDirc + f'/MRL_distribution_curves_{sim}.png',
    include_legend=True,
    grid=True,
    spline_curvature=True,
)
#plot a small verticle line at 99th percentile of MRL_200_values
percentile_99_MRL_200 = np.percentile(MRL_200_values, 99)
MRL_ax.axvline(percentile_99_MRL_200, color='green', linestyle='--', label='99th Percentile (200 Non-Centrals)')
MRL_ax.legend()

In [ ]:
def overlayMRLAndMRLRandom(listGG : list[tuple[ListGalaxyGroup, str]]):
    overlayMRLPlotter = AstroPlotter()
    overlayMRLFig, overlayMRLAx = overlayMRLPlotter.create_figure()
    
    for i, (listGalaxyGroup, label) in enumerate(listGG):
        MRL_values = listGalaxyGroup.compute_probablity_distribution_of_MRL_directionality(parallelize=False)
        MRL_binned, MRL_bin_edges, MRL_errorbars = ListGalaxyGroup.get_histogram_bins(MRL_values, bin_size=0.005, binLow=0, binHigh=1, errorbarType='poisson')
        
        random_MRL_values = listGalaxyGroup.compute_MRL_random_distribution_curves_for_LGG(parallelize=False, num_samples=10000)
        random_MRL_bins, random_MRL_bin_centers, random_MRL_errorbars = ListGalaxyGroup.get_histogram_bins(random_MRL_values, bin_size=0.005, binLow=0, binHigh=1, errorbarType='poisson')
        
        overlayMRLPlotter.scatter_plot(
            MRL_bin_edges, 
            MRL_binned,
            # errorBars = MRL_errorbars,
            ax=overlayMRLAx, # Use the same axis for overlay
            # ylim = (0, 0.01),
            label=f"MRL Directionality ({listGalaxyGroup.getNumGalaxyGroups()} Galaxy Groups)",
            output_filename=None,  # Disable saving for combined plot
            grid=True
        )
        #plot a small verticle line at 99th percentile of radnom MRL_values
        percentile_99_MRL = np.percentile(random_MRL_values, 99)
        overlayMRLAx.axvline(percentile_99_MRL, linestyle='--', label=f'99th Percentile')
        
        overlayMRLPlotter.scatter_plot(
            random_MRL_bin_centers, 
            random_MRL_bins,
            # errorBars = random_MRL_errorbars,
            ax=overlayMRLAx,  # Use the same axis for overlay
            xlabel='MRL Directionality',
            ylabel='Probability Density',
            title=f'MRL Distribution Curves vs Random for {sim}',
            # ylim = (0, 0.01),
            label=f"Random MRL Directionality ({len(random_MRL_values)} Samples)",
            include_legend=True,
            # output_filename=scratchPlotDirc + f'/MRL_distribution_curves_overlay_{sim}.png',
            output_filename=None,
            grid=True,
        )
        print(f"99th percentile MRL: {percentile_99_MRL}")
        print(f"Overall number of MRL values above 99th percentile: {np.sum(MRL_values > percentile_99_MRL)} out of {len(MRL_values)}")
        
        print(f"len listGG: {len(listGG)}, len MRL_values: {len(MRL_values)}, len random_MRL_values: {len(random_MRL_values)}")
        
        # save into .txt file:
        # In table: galaxy id, num of members, mass of cluster, MRL value of that projection, fraction less than the MRL I measured
        with open(scratchPlotDirc + f'/MRL_values_and_random_comparison_{label}.txt', 'w') as f:
            f.write("GalaxyGroupID\tNumMembers\tClusterMass\tMRLValue\tFractionLessThanMRL\n")
            for gg in listGalaxyGroup.getAllGalaxyGroups():
                gg_id = gg.getGroupID()
                num_members = gg.getNumSubhalos()
                cluster_mass = gg.getMCrit200()
                MRL_value = MRL_values[gg_id] if gg_id in MRL_values else 0
                fraction_less_than_MRL = np.sum(random_MRL_values < MRL_value) / len(random_MRL_values)
                f.write(f"{gg_id}\t{num_members}\t{cluster_mass}\t{MRL_value}\t{fraction_less_than_MRL}\n")
    overlayMRLAx.legend()
    overlayMRLPlotter.save_figure(overlayMRLFig, scratchPlotDirc + f'/MRL_distribution_curves_overlay_{sim}.png')
    
    #append into overall .txt file:
    with open(scratchPlotDirc + f'/MRL_values_and_random_comparison_overall.txt', 'a') as f:
        f.write(f"Simulation: {sim}\n")
        for i, (listGalaxyGroup, label) in enumerate(listGG):
            with open(scratchPlotDirc + f'/MRL_values_and_random_comparison_{label}.txt', 'r') as g:
                lines = g.readlines()
                for line in lines[1:]:  # Skip header line
                    f.write(line)

In [ ]:
#plot the MRL values for all galaxy groups and highlight those above the 99th percentile for the 200 non-centrals case
list_of_galaxy_groups_gt13 = list_of_galaxy_groups.getFilterSubhalos(minGGMass=1e13)
MRL_gt13_values = list_of_galaxy_groups_gt13.compute_probablity_distribution_of_MRL_directionality(parallelize=False)
MRL_gt13_bins, MRL_gt13_bin_centers, MRL_gt13_errorbars = ListGalaxyGroup.get_histogram_bins(MRL_gt13_values, bin_size=0.005, binLow=0, binHigh=1, errorbarType='poisson')

random_MRL_values_gt13 = list_of_galaxy_groups_gt13.compute_MRL_random_distribution_curves_for_LGG(parallelize=False, num_samples=10000)
random_MRL_gt13_bins, random_MRL_gt13_bin_centers, random_MRL_gt13_errorbars = ListGalaxyGroup.get_histogram_bins(random_MRL_values_gt13, bin_size=0.005, binLow=0, binHigh=1, errorbarType='poisson')

MRL_gt13_plotter = AstroPlotter()
MRL_gt13_fig, MRL_gt13_ax = MRL_gt13_plotter.create_figure()
#plot the random MRL distribution on the same plot
MRL_gt13_plotter.scatter_plot(
    random_MRL_gt13_bin_centers, 
    random_MRL_values_gt13,
    # errorBars = random_MRL_gt13_errorbars,
    ax=MRL_gt13_ax,  # Use the same axis for overlay
    # ylim = (0, 0.01),
    label="Random Distribution (200 Non-Centrals)",
    overlay_color='gray',
    alpha=0,
    output_filename=None,
    include_legend=True,
    grid=True,
    spline_curvature=True,
)
MRL_gt13_plotter.scatter_plot(
    MRL_gt13_bin_centers, 
    MRL_gt13_values,
    # errorBars = MRL_gt13_errorbars,
    ax=MRL_gt13_ax, # Use the same axis for overlay
    xlabel='MRL Directionality',
    ylabel='Probability Density',
    title=f'MRL Directionality for Galaxy Groups with $M_{{200}}$>1e13 Msun in {sim}',
    # ylim = (0, 0.01),
    label="Galaxy Groups",
    output_filename=scratchPlotDirc + f'/MRL_gt13_with_random_{sim}.png',
    grid=True,
)

In [ ]:
#alternatively
overlayMRLAndMRLRandom([list_of_galaxy_groups_gt13])

In [ ]:
#compute for each group (13-13.5, 13.5-14, 14-14.5, 14.5-15, >15)
overlayMRLAndMRLRandom([filtered_13_13p5_list_of_galaxy_groups, filtered_13p5_14_list_of_galaxy_groups, filtered_14_14p5_list_of_galaxy_groups, filtered_14p5_15_list_of_galaxy_groups, filtered_gt15_list_of_galaxy_groups])

## TBD